# Karachi Air Quality Forecasting - Phases 2 and 3
Phase 2: cleaning and features. Phase 3: exploration (EDA).
Needs the file `karachi_aq_history.csv` from `backfill.py` in the same folder.

## Phase 2: Cleaning and features

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Step 1: Load the data and check it

In [ ]:
table = pd.read_csv("karachi_aq_history.csv")

print("Rows and columns:", table.shape)
print(table.head())
print(table.info())

### Step 2: Fix the time column and remove duplicates

In [ ]:
# make the time column a real date and time
table["time"] = pd.to_datetime(table["time"])

# put rows in time order
table = table.sort_values("time")

# remove rows that have the same time twice
rows_before = len(table)
table = table.drop_duplicates(subset="time")
rows_after = len(table)

print("Duplicate rows removed:", rows_before - rows_after)

### Step 3: Fill missing values

In [ ]:
# how many missing values in each column
print(table.isna().sum())

In [ ]:
# fill small gaps using the values before and after
table = table.set_index("time")
table = table.interpolate(method="time")
table = table.reset_index()

# check again, it should now be 0 (or very small)
print(table.isna().sum())

### Step 4: Add time columns

In [ ]:
table["hour"] = table["time"].dt.hour
table["day_of_week"] = table["time"].dt.dayofweek    # 0 = Monday, 6 = Sunday
table["month"] = table["time"].dt.month

# weekend = Saturday or Sunday
table["is_weekend"] = 0
table.loc[table["day_of_week"] >= 5, "is_weekend"] = 1

# night = 10 PM to 5 AM
table["is_night"] = 0
table.loc[(table["hour"] >= 22) | (table["hour"] <= 5), "is_night"] = 1

# rush hours = 7 to 10 AM and 5 to 8 PM
table["is_rush_hour"] = 0
table.loc[(table["hour"] >= 7) & (table["hour"] <= 10), "is_rush_hour"] = 1
table.loc[(table["hour"] >= 17) & (table["hour"] <= 20), "is_rush_hour"] = 1

print(table[["time", "hour", "day_of_week", "is_weekend", "is_night", "is_rush_hour"]].head())

### Step 5: Add past-value columns (lags) and rolling averages
Each row only uses values from the PAST, so the model never sees the future.

In [ ]:
# PM2.5 value 1, 3 and 24 hours ago
table["pm2_5_lag1"] = table["pm2_5"].shift(1)
table["pm2_5_lag3"] = table["pm2_5"].shift(3)
table["pm2_5_lag24"] = table["pm2_5"].shift(24)

# average of the last 3 and last 24 hours (not including the current hour)
past_pm = table["pm2_5"].shift(1)
table["pm2_5_avg3"] = past_pm.rolling(3).mean()
table["pm2_5_avg24"] = past_pm.rolling(24).mean()

# the first 24 rows have no past values, so remove them
table = table.dropna()

print("Rows and columns after cleaning:", table.shape)
print(table.head())

### Step 6: Save the clean table

In [ ]:
table.to_csv("clean_data.csv", index=False)
print("Saved clean_data.csv")

## Phase 3: Exploration (EDA)

### Summary numbers

In [ ]:
print(table[["pm2_5", "pm10", "nitrogen_dioxide", "ozone", "temperature_2m", "relative_humidity_2m", "wind_speed_10m"]].describe())

### Chart 1: PM2.5 over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(table["time"], table["pm2_5"])
plt.axhline(35, color="red", linestyle="--", label="35 limit (example)")
plt.title("PM2.5 in Karachi over time")
plt.xlabel("Time")
plt.ylabel("PM2.5 (ug/m3)")
plt.legend()
plt.show()

**Interpretation:** (write 2 or 3 sentences after you see the chart: is there a trend? are there big spikes?)

### Chart 2: Average PM2.5 by hour of day

In [ ]:
hour_average = table.groupby("hour")["pm2_5"].mean()

plt.figure(figsize=(10, 4))
plt.bar(hour_average.index, hour_average.values)
plt.title("Average PM2.5 by hour of day")
plt.xlabel("Hour (0 to 23)")
plt.ylabel("Average PM2.5 (ug/m3)")
plt.show()

**Interpretation:** (which hours are highest and lowest? does it match rush hours?)

### Chart 3: Average PM2.5 by day of week

In [ ]:
day_average = table.groupby("day_of_week")["pm2_5"].mean()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

plt.figure(figsize=(8, 4))
plt.bar(day_names, day_average.values)
plt.title("Average PM2.5 by day of week")
plt.xlabel("Day")
plt.ylabel("Average PM2.5 (ug/m3)")
plt.show()

**Interpretation:** (is the weekend different from weekdays?)

### Chart 4: PM2.5 versus wind speed and humidity

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(table["wind_speed_10m"], table["pm2_5"], s=5)
plt.title("PM2.5 vs wind speed")
plt.xlabel("Wind speed (km/h)")
plt.ylabel("PM2.5 (ug/m3)")

plt.subplot(1, 2, 2)
plt.scatter(table["relative_humidity_2m"], table["pm2_5"], s=5)
plt.title("PM2.5 vs humidity")
plt.xlabel("Humidity (%)")
plt.ylabel("PM2.5 (ug/m3)")

plt.show()

**Interpretation:** (does more wind mean less PM2.5? what about humidity?)

### Chart 5: Correlation heatmap

In [ ]:
small_table = table[["pm2_5", "pm10", "nitrogen_dioxide", "ozone", "temperature_2m", "relative_humidity_2m", "wind_speed_10m", "pm2_5_lag1", "pm2_5_lag24"]]

plt.figure(figsize=(9, 7))
sns.heatmap(small_table.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation between variables")
plt.show()

**Interpretation:** (which variables are most related to PM2.5? is lag1 very high? what does that mean for forecasting?)

## Notes for the report
- Data comes from the CAMS model through Open-Meteo, not from ground sensors.
- The 35 line on the charts is an example limit. Confirm the exact standard you cite.
- Next: Phase 4 (SARIMA) will use `clean_data.csv`.